# Import Libraries

In [5]:
import ast
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load 3 CSV files

In [6]:
train_raw = pd.read_csv(r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Training_clean.csv",
                        usecols=["id", "acts", "emotions", "utterances", "context_state"])

test_raw = pd.read_csv(r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Testing_clean.csv",
                       usecols=["id", "acts", "emotions", "utterances", "context_state"])

val_raw = pd.read_csv(r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Validation_clean.csv",
                      usecols=["id", "acts", "emotions", "utterances", "context_state"])

print("Loaded → Train:", len(train_raw))
print("Loaded → Test :", len(test_raw))
print("Loaded → Val  :", len(val_raw))

Loaded → Train: 11118
Loaded → Test : 1000
Loaded → Val  : 1000


# Feature & Target Creation

In [8]:
def prepare_data(df):
    rows = []

    for _, row in df.iterrows():
        cid = row["id"]
        context = str(row["context_state"])

        utterances = ast.literal_eval(row["utterances"]) if isinstance(row["utterances"], str) else row["utterances"]
        emotions   = ast.literal_eval(row["emotions"])   if isinstance(row["emotions"], str) else row["emotions"]
        acts       = ast.literal_eval(row["acts"])       if isinstance(row["acts"], str) else row["acts"]

        for utt, emo, act in zip(utterances, emotions, acts):
            rows.append({
                "utterance_text": utt,
                "emotion": str(emo),
                "act": str(act),
                "context": context
            })

    return pd.DataFrame(rows)


train = prepare_data(train_raw)
valid = prepare_data(val_raw)
test  = prepare_data(test_raw)

# Dataset Split Sizes (Train / Validation / Test)

In [9]:
print("\nTrain shape:", train.shape)
print("Valid shape:", valid.shape)
print("Test shape :", test.shape)


Train shape: (87170, 4)
Valid shape: (8069, 4)
Test shape : (7740, 4)


# Dataset Column Names

In [10]:
print(train.columns.tolist())

['utterance_text', 'emotion', 'act', 'context']


# TASK CONFIG

In [12]:
tasks = {
    "Emotion": "emotion",
    "Act": "act",
    "Context": "context"
}

# Dictionary to store all results for final summary

In [13]:
results_summary = {}

# Model Train

In [14]:
for task_name, target_col in tasks.items():

    print("\n" + "="*60)
    print(f"TASK: {task_name}")
    print("="*60)

    X_train = train["utterance_text"]
    y_train = train[target_col]

    X_valid = valid["utterance_text"]
    y_valid = valid[target_col]

    X_test  = test["utterance_text"]
    y_test  = test[target_col]

    # ================= SVM =================
    svm_model = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LinearSVC(random_state=42))
    ])

    svm_model.fit(X_train, y_train)

    test_pred_svm = svm_model.predict(X_test)

    print("\n----- SVM -----")
    print("Train Accuracy:", accuracy_score(y_train, svm_model.predict(X_train)))
    print("Valid Accuracy:", accuracy_score(y_valid, svm_model.predict(X_valid)))
    print("Test Accuracy :", accuracy_score(y_test, test_pred_svm))
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred_svm))


    # ================= RANDOM FOREST =================
    rf_model = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000)),
        ("clf", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_model.fit(X_train, y_train)

    test_pred_rf = rf_model.predict(X_test)

    print("\n----- Random Forest -----")
    print("Train Accuracy:", accuracy_score(y_train, rf_model.predict(X_train)))
    print("Valid Accuracy:", accuracy_score(y_valid, rf_model.predict(X_valid)))
    print("Test Accuracy :", accuracy_score(y_test, test_pred_rf))
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred_rf))


    # SAVE RESULTS
    results_summary[task_name] = {
        "SVM": {
            "Train": accuracy_score(y_train, svm_model.predict(X_train)),
            "Valid": accuracy_score(y_valid, svm_model.predict(X_valid)),
            "Test": accuracy_score(y_test, test_pred_svm)
        },
        "RandomForest": {
            "Train": accuracy_score(y_train, rf_model.predict(X_train)),
            "Valid": accuracy_score(y_valid, rf_model.predict(X_valid)),
            "Test": accuracy_score(y_test, test_pred_rf)
        }
    }


TASK: Emotion

----- SVM -----
Train Accuracy: 0.8770792703911896
Valid Accuracy: 0.8957739496839757
Test Accuracy : 0.8421188630490956

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.97      0.91      6321
           1       0.56      0.08      0.13       118
           2       0.75      0.06      0.12        47
           3       1.00      0.12      0.21        17
           4       0.64      0.37      0.47      1019
           5       0.54      0.07      0.12       102
           6       0.48      0.13      0.20       116

    accuracy                           0.84      7740
   macro avg       0.69      0.26      0.31      7740
weighted avg       0.82      0.84      0.81      7740


----- Random Forest -----
Train Accuracy: 0.9680624067913273
Valid Accuracy: 0.8978807782872723
Test Accuracy : 0.8410852713178295

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.

# FINAL SUMMARY

In [15]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

for task, models in results_summary.items():
    print(f"\nTask: {task}")
    for model, scores in models.items():
        print(f"  {model}:")
        print(f"    Train: {scores['Train']*100:.2f}%")
        print(f"    Valid: {scores['Valid']*100:.2f}%")
        print(f"    Test : {scores['Test']*100:.2f}%")


FINAL SUMMARY

Task: Emotion
  SVM:
    Train: 87.71%
    Valid: 89.58%
    Test : 84.21%
  RandomForest:
    Train: 96.81%
    Valid: 89.79%
    Test : 84.11%

Task: Act
  SVM:
    Train: 78.28%
    Valid: 67.51%
    Test : 71.58%
  RandomForest:
    Train: 96.77%
    Valid: 72.76%
    Test : 76.58%

Task: Context
  SVM:
    Train: 71.56%
    Valid: 65.49%
    Test : 62.52%
  RandomForest:
    Train: 95.54%
    Valid: 68.12%
    Test : 67.00%


# DailyDialog Dataset Sample Inspection (Utterances, Acts, Emotions, Context)

In [17]:
for i in range(5):

    print("\n" + "="*60)
    print("SAMPLE:", i)
    print("="*60)

    print("\nUTTERANCE:")
    print(train["utterance_text"].iloc[i])

    print("\nEMOTION LABEL:")
    print(train["emotion"].iloc[i])

    print("\nACT LABEL:")
    print(train["act"].iloc[i])

    print("\nCONTEXT:")
    print(train["context"].iloc[i])


SAMPLE: 0

UTTERANCE:
Say , Jim , how about going for a few beers after dinner ?

EMOTION LABEL:
0

ACT LABEL:
3

CONTEXT:
achievement

SAMPLE: 1

UTTERANCE:
You know that is tempting but is really not good for our fitness .

EMOTION LABEL:
0

ACT LABEL:
4

CONTEXT:
achievement

SAMPLE: 2

UTTERANCE:
What do you mean ? It will help us to relax .

EMOTION LABEL:
0

ACT LABEL:
2

CONTEXT:
achievement

SAMPLE: 3

UTTERANCE:
Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?

EMOTION LABEL:
0

ACT LABEL:
2

CONTEXT:
achievement

SAMPLE: 4

UTTERANCE:
I guess you are right.But what shall we do ? I don't feel like sitting at home .

EMOTION LABEL:
0

ACT LABEL:
2

CONTEXT:
achievement
